In [1]:
import nltk
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [27]:
import re 
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report,accuracy_score,confusion_matrix



In [3]:
data= pd.read_csv('IMDB Dataset.csv')
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
print(data.shape)
print(data.columns)
print(data['sentiment'].value_counts())

(50000, 2)
Index(['review', 'sentiment'], dtype='object')
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [5]:
data.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
data.duplicated().sum()

418

In [7]:
data.drop_duplicates(inplace=True)

In [8]:
data['sentiment'].value_counts()

sentiment
positive    24884
negative    24698
Name: count, dtype: int64

In [9]:
#select one review
sample_review= data.loc[0,"review"]
print(f"original review: {sample_review}")
print("Sentimet label:",data.loc[0,"sentiment"])

original review: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show 

## NLP Preprocessing Pipeline

In [10]:
#text cleaning

def clean_text(text):
    text= text.lower()
    #remove html tags
    text= re.sub(r"<[^>]+>"," ",text)
    #remove urls
    text=re.sub(r"https?://\s+|www\.\s+"," ",text)
    #remove extra spaces
    text= re.sub(r"\s+", " ",text)
    #keep letters and spaces only
    text=re.sub(r"[^a-z\s]", '',text).strip()
    return text

In [11]:
#test the function
test_review = """
The movie was AMAZING!!! <br />
I did NOT like the ending. Visit https://example.com
"""

print("Before cleaning:")
print(test_review)

print("\nAfter cleaning:")
print(clean_text(test_review))

Before cleaning:

The movie was AMAZING!!! <br />
I did NOT like the ending. Visit https://example.com


After cleaning:
the movie was amazing i did not like the ending visit httpsexamplecom


In [12]:
data['cleaned_review']= data['review'].apply(clean_text)
data.head()

,review,sentiment,cleaned_review
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production the filming tech...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,negative,basically theres a family where a little boy j...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love in the time of money is a ...


In [13]:
stemmer= PorterStemmer()
lemmatizer= WordNetLemmatizer()
nltk.download("punkt_tab")
data['tokens']= data['cleaned_review'].apply(word_tokenize)
display(data[["cleaned_review", "tokens", "sentiment"]].head())

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,cleaned_review,tokens,sentiment
0,one of the other reviewers has mentioned that ...,"[one, of, the, other, reviewers, has, mentione...",positive
1,a wonderful little production the filming tech...,"[a, wonderful, little, production, the, filmin...",positive
2,i thought this was a wonderful way to spend ti...,"[i, thought, this, was, a, wonderful, way, to,...",positive
3,basically theres a family where a little boy j...,"[basically, theres, a, family, where, a, littl...",negative
4,petter matteis love in the time of money is a ...,"[petter, matteis, love, in, the, time, of, mon...",positive


In [20]:
#lemmatization with pos
from nltk import pos_tag
from nltk.corpus import wordnet

def get_wordnet_pos(tag):
    if tag.startswith("j"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

def lemmatize_tokens(tokens):
    tagged_token= pos_tag(tokens)

    return [
        lemmatizer.lemmatize(word,get_wordnet_pos(tag))
        for word ,tag in tagged_token
    ]


In [21]:
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("omw-1.4")
data["lemmatized_tokens"]= data['tokens'].apply(lemmatize_tokens)
data['lemmatized_review']= data['lemmatized_tokens'].apply(
    lambda tokens: " ".join(tokens)
)

display(data[['cleaned_review','tokens','lemmatized_review','sentiment']])

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,cleaned_review,tokens,lemmatized_review,sentiment
0,one of the other reviewers has mentioned that ...,"[one, of, the, other, reviewers, has, mentione...",one of the other reviewer have mention that af...,positive
1,a wonderful little production the filming tech...,"[a, wonderful, little, production, the, filmin...",a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,"[i, thought, this, was, a, wonderful, way, to,...",i think this be a wonderful way to spend time ...,positive
3,basically theres a family where a little boy j...,"[basically, theres, a, family, where, a, littl...",basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,"[petter, matteis, love, in, the, time, of, mon...",petter matteis love in the time of money be a ...,positive
...,...,...,...,...
49995,i thought this movie did a down right good job...,"[i, thought, this, movie, did, a, down, right,...",i think this movie do a down right good job it...,positive
49996,bad plot bad dialogue bad acting idiotic direc...,"[bad, plot, bad, dialogue, bad, acting, idioti...",bad plot bad dialogue bad act idiotic direct t...,negative
49997,i am a catholic taught in parochial elementary...,"[i, am, a, catholic, taught, in, parochial, el...",i be a catholic taught in parochial elementary...,negative
49998,im going to have to disagree with the previous...,"[im, going, to, have, to, disagree, with, the,...",im go to have to disagree with the previous co...,negative


In [ ]:
data[["lemmatized_review", "sentiment"]].isnull().sum()

In [22]:
from sklearn.preprocessing import LabelEncoder

encoder= LabelEncoder()
data['sentiment_encoded']= encoder.fit_transform(data['sentiment'])

In [23]:
#train test split

x= data['lemmatized_review']
y=data['sentiment_encoded']

x_train,x_test,y_train,y_test= train_test_split(
    x,y,random_state=42,stratify=y
)
print("training samples", len(x_train))
print("testing samples",len(x_test))

training samples 37186
testing samples 12396


In [25]:
vectorizer= TfidfVectorizer(
    tokenizer=word_tokenize,
    token_pattern=None,
    lowercase=False
)
x_train_tfidf= vectorizer.fit_transform(x_train)
x_test_tfidf= vectorizer.transform(x_test)
print("Training matrix shape:", x_train_tfidf.shape)
print("Testing matrix shape:", x_test_tfidf.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))


Training matrix shape: (37186, 125869)
Testing matrix shape: (12396, 125869)
Vocabulary size: 125869


In [29]:
model= MultinomialNB()
model.fit(x_train_tfidf,y_train)
y_pred= model.predict(x_test_tfidf)


In [30]:
#evalution
print("Accuracy :", accuracy_score(y_test,y_pred))
print("classification report : \n")
print(classification_report(y_test,y_pred,zero_division=0))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.8606001936108422
classification report : 

              precision    recall  f1-score   support

           0       0.84      0.88      0.86      6175
           1       0.88      0.84      0.86      6221

    accuracy                           0.86     12396
   macro avg       0.86      0.86      0.86     12396
weighted avg       0.86      0.86      0.86     12396


Confusion Matrix:
[[5453  722]
 [1006 5215]]


In [34]:
from sklearn.linear_model import LinearRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score


from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

# 1. Define the models
models = {
    "MultinomialNB": model,
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "LinearSVC": LinearSVC()
}

results = []

# 2. Train and evaluate each model
for name, clf in models.items():

    # The first model is already trained
    if name != "MultinomialNB":
        clf.fit(x_train_tfidf, y_train)

    predictions = clf.predict(x_test_tfidf)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "F1-Score": f1_score(
            y_test,
            predictions,
            pos_label=1
        )
    })

# 3. Compare results
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="F1-Score",
    ascending=False
)

display(results_df.round(4))


,Model,Accuracy,F1-Score
2,LinearSVC,0.8971,0.8985
1,LogisticRegression,0.8905,0.8926
0,MultinomialNB,0.8606,0.8579


In [37]:


# Create a new TF-IDF vectorizer
vectorizer_ngrams = TfidfVectorizer(
    tokenizer=word_tokenize,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    max_features=100000
)

# Learn TF-IDF features from training data only
X_train_ngrams = vectorizer_ngrams.fit_transform(x_train)

# Transform test data using the same vocabulary
X_test_ngrams = vectorizer_ngrams.transform(x_test)

# Train LinearSVC
model_ngrams = LinearSVC()
model_ngrams.fit(X_train_ngrams, y_train)

# Evaluate
y_pred_ngrams = model_ngrams.predict(X_test_ngrams)

print("Accuracy:", round(
    accuracy_score(y_test, y_pred_ngrams), 4
))

print("F1-Score:", round(
    f1_score(y_test, y_pred_ngrams, pos_label=1), 4
))

print("Vocabulary size:", len(vectorizer_ngrams.vocabulary_))


Accuracy: 0.9084
F1-Score: 0.9095
Vocabulary size: 100000


In [53]:
def predict_sentiment(review):
    #clean the input text
    cleaned= clean_text(review)
    tokens= word_tokenize(cleaned)
    lemmatized= lemmatize_tokens(tokens)
    #convert into tfidf features
    processed_review= " ".join(lemmatized)
    review_tfidf= vectorizer_ngrams.transform([processed_review])

    prediction = model_ngrams.predict(review_tfidf)[0]
    if prediction == 0:
        return "Negative"
    else:
        return "Positive"

    

test_reviews = [
    "This movie was absolutely amazing!",
    "This movie was terrible and boring.",
    "The acting was good but the story was disappointing.",
    "I did not enjoy this movie at all.",
    "I expected a lot, but this film was a complete waste of time."
]

for review in test_reviews:
    prediction = predict_sentiment(review)
    print("Review:", review)
    print("Predicted sentiment:", prediction)
    print("-" * 60)


Review: This movie was absolutely amazing!
Predicted sentiment: Positive
------------------------------------------------------------
Review: This movie was terrible and boring.
Predicted sentiment: Negative
------------------------------------------------------------
Review: The acting was good but the story was disappointing.
Predicted sentiment: Negative
------------------------------------------------------------
Review: I did not enjoy this movie at all.
Predicted sentiment: Negative
------------------------------------------------------------
Review: I expected a lot, but this film was a complete waste of time.
Predicted sentiment: Negative
------------------------------------------------------------


In [54]:
#save models
import joblib

joblib.dump(model_ngrams,"sentiment_model.pkl")
joblib.dump(vectorizer_ngrams,"tfidf_vectorizer.pkl")

print("mdel and vectorizer saved successfully")

mdel and vectorizer saved successfully
